# Role Routing and Task Assignment
Companion notebook for the **Role Routing and Task Assignment** section of Chapter 8.

Start with **rule-based routing**; add model-based routing only where rules run out. Rules are easy to test and reason about.

A tiny stand-in for the model-based fallback, so `route_request` below is fully runnable without any API key:

In [1]:
def classify_with_router_model(text: str, context: dict) -> str:
    """Stand-in for a real router-model call, used only when the rules don't match."""
    return "generalist_agent"

This is the book's `route_request` function, unchanged:

In [2]:
def route_request(text, context):
    if context.get("needs_human_approval"):
        return "human_review"
    if "refund" in text.lower():
        return "billing_agent"
    if "password" in text.lower():
        return "account_agent"
    return classify_with_router_model(text, context)

In [3]:
cases = [
    ("I need help with a billing refund.", {}),
    ("Please reset my password.", {}),
    ("Can you cancel my subscription?", {"needs_human_approval": True}),
    ("Tell me something interesting about your product roadmap.", {}),
]

for text, context in cases:
    print(f"{route_request(text, context):16s} <- {text!r} {context}")

billing_agent    <- 'I need help with a billing refund.' {}
account_agent    <- 'Please reset my password.' {}
human_review     <- 'Can you cancel my subscription?' {'needs_human_approval': True}
generalist_agent <- 'Tell me something interesting about your product roadmap.' {}


Guard the handoff itself, not just the routing decision: a confidence threshold before trusting a model-based route, a fallback route when confidence is low, and an **idempotency key** so a retried handoff doesn't trigger the same side effect twice.

In [4]:
CONFIDENCE_THRESHOLD = 0.6
processed_idempotency_keys: set[str] = set()


def route_with_confidence(text: str, context: dict, confidence: float) -> str:
    if confidence < CONFIDENCE_THRESHOLD:
        return "human_review"  # fallback route when the model isn't sure
    return route_request(text, context)


def handle_once(idempotency_key: str, action):
    """Runs `action` only if this idempotency_key hasn't been processed before."""
    if idempotency_key in processed_idempotency_keys:
        return {"status": "skipped_duplicate"}
    processed_idempotency_keys.add(idempotency_key)
    return action()


print(route_with_confidence("vague request", {}, confidence=0.3))
print(handle_once("refund-req-42", lambda: {"status": "processed"}))
print(handle_once("refund-req-42", lambda: {"status": "processed"}))  # retried handoff, same key

human_review
{'status': 'processed'}
{'status': 'skipped_duplicate'}
